# Hunyuan3D-2: Image-to-3D with Texture (RunPod)

Generate a textured 3D model from a single image using Tencent's Hunyuan3D-2.

**RunPod Setup:**
- GPU: RTX 4090 (24 GB VRAM)
- Template: **RunPod Pytorch 2.4.0** (py3.11-cuda12.4.1)
- Container Disk: 40 GB
- Volume Disk: 50 GB

**Why this config:**
- CUDA 12.4 required for texture rasterizer C++ compilation
- We install PyTorch 2.6.0+cu124 to satisfy transformers >= 2.6 security check (CVE-2025-32434)
- numpy pinned to <2.0 for trimesh compatibility
- Pillow upgraded to >=10.4 to fix `JpegImageFile._im` AttributeError
- VAE weights ship as `.bin` (not `.safetensors`); the fallback warning is suppressed

**Run order:** Steps 1-3 (first time only) → restart kernel → Steps 4-10

## Step 1: Redirect caches to /workspace

In [ ]:
import os

!mkdir -p /workspace/.cache /workspace/tmp
!test -L /root/.cache || (rsync -a /root/.cache/ /workspace/.cache/ 2>/dev/null; rm -rf /root/.cache; ln -sf /workspace/.cache /root/.cache)

os.environ["HF_HOME"] = "/workspace/.cache/huggingface"
os.environ["HF_HUB_CACHE"] = "/workspace/.cache/huggingface/hub"
os.environ["HUGGINGFACE_HUB_CACHE"] = "/workspace/.cache/huggingface/hub"
os.environ["TORCH_HOME"] = "/workspace/.cache/torch"
os.environ["PIP_CACHE_DIR"] = "/workspace/.cache/pip"
os.environ["TMPDIR"] = "/workspace/tmp"

!df -h / /workspace
print("\nCaches redirected to /workspace.")

## Step 2: Install PyTorch 2.6.0 + cu124 & dependencies

In [ ]:
# Upgrade typing_extensions first (torch 2.6 needs TypeIs support)
# Upgrade Pillow to fix JpegImageFile._im AttributeError
!pip install --no-cache-dir "typing_extensions>=4.10" "Pillow>=10.4" 2>&1 | tail -2

# Install PyTorch 2.6.0 + cu124
# - CUDA 12.4 needed for texture rasterizer compilation
# - torch >= 2.6 needed for transformers security check (CVE-2025-32434)
!pip install --no-cache-dir \
    torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 \
    --index-url https://download.pytorch.org/whl/cu124 \
    2>&1 | tail -5

!python3 -c "import torch; print(f'PyTorch: {torch.__version__}, CUDA: {torch.version.cuda}')"
!python3 -c "import torchvision; print(f'torchvision: {torchvision.__version__}')"
!python3 -c "from PIL import Image; print(f'Pillow: {Image.__version__}')"

In [ ]:
# Clone repo
import os
os.chdir("/workspace")
if not os.path.exists("/workspace/Hunyuan3D-2"):
    !git clone https://github.com/Tencent/Hunyuan3D-2.git
else:
    print("Repo already cloned.")
os.chdir("/workspace/Hunyuan3D-2")
print(f"Working directory: {os.getcwd()}")

In [ ]:
# Install project dependencies
!pip install --no-cache-dir -r /workspace/Hunyuan3D-2/requirements.txt matplotlib 2>&1 | tail -5

# Pin numpy <2.0 to avoid trimesh/scipy compatibility issues
!pip install --no-cache-dir "numpy>=1.26.4,<2.0" 2>&1 | tail -3

# Re-pin torch 2.6.0+cu124 in case requirements.txt downgraded it
!pip install --no-cache-dir \
    torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 \
    --index-url https://download.pytorch.org/whl/cu124 \
    2>&1 | tail -3

# Verify key versions
!python3 -c "import torch; print('torch:', torch.__version__)"
!python3 -c "import torchvision; print('torchvision:', torchvision.__version__)"
!python3 -c "import transformers; print('transformers:', transformers.__version__)"
!python3 -c "import numpy; print('numpy:', numpy.__version__)"
!python3 -c "import diffusers; print('diffusers:', diffusers.__version__)"
!python3 -c "import scipy; print('scipy:', scipy.__version__)"
!python3 -c "import trimesh; print('trimesh:', trimesh.__version__)"

## Step 3: Build C++ extensions & restart

In [ ]:
# Build texture generation C++ extensions (requires CUDA 12.4)
!cd /workspace/Hunyuan3D-2/hy3dgen/texgen/custom_rasterizer && pip install . 2>&1 | tail -3
!cd /workspace/Hunyuan3D-2/hy3dgen/texgen/differentiable_renderer && pip install . 2>&1 | tail -3
print("\nBuild complete. RESTART THE KERNEL now (Kernel > Restart), then run from Step 4.")

---
## Step 4: Setup after kernel restart

Upload `banana.jpg` to `/workspace/Hunyuan3D-2/` via the Jupyter file browser, then run this cell.

In [ ]:
import os
import torch

# Restore working directory and env vars after kernel restart
os.chdir("/workspace/Hunyuan3D-2")
os.environ["HF_HOME"] = "/workspace/.cache/huggingface"
os.environ["HF_HUB_CACHE"] = "/workspace/.cache/huggingface/hub"
os.environ["HUGGINGFACE_HUB_CACHE"] = "/workspace/.cache/huggingface/hub"
os.environ["TORCH_HOME"] = "/workspace/.cache/torch"
os.environ["TMPDIR"] = "/workspace/tmp"

print(f"PyTorch: {torch.__version__}, CUDA: {torch.version.cuda}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
print(f"Working dir: {os.getcwd()}")

from PIL import Image
import matplotlib.pyplot as plt

INPUT_IMAGE = "/workspace/Hunyuan3D-2/frame.jpg"
OUTPUT_PATH = "/workspace/Hunyuan3D-2/output/model.glb"

img = Image.open(INPUT_IMAGE)
plt.figure(figsize=(6, 6))
plt.imshow(img)
plt.axis("off")
plt.title("Input Image")
plt.show()
print(f"Image size: {img.size}")

## Step 5: Remove background & generate 3D mesh

The official examples use `BackgroundRemover` to strip backgrounds before generation. This produces much better results.

In [ ]:
from PIL import Image
from hy3dgen.rembg import BackgroundRemover
from hy3dgen.shapegen import Hunyuan3DDiTFlowMatchingPipeline

# Remove background (critical for good results)
raw_image = Image.open(INPUT_IMAGE)
needs_rembg = raw_image.mode == "RGB"  # JPGs are always RGB, PNGs may have alpha
image = raw_image.convert("RGBA")
if needs_rembg:
    rembg = BackgroundRemover()
    image = rembg(image)

# Show preprocessed image
plt.figure(figsize=(6, 6))
plt.imshow(image)
plt.axis("off")
plt.title("After Background Removal")
plt.show()

# Generate 3D mesh
shape_pipeline = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained("tencent/Hunyuan3D-2")
mesh = shape_pipeline(image=image)[0]
print(f"Mesh: {len(mesh.vertices)} vertices, {len(mesh.faces)} faces")

## Step 6: Apply texture

The texture pipeline defaults to `device='cpu'` internally, which causes 0% GPU utilization. We patch the source file and override the device to force GPU execution.

In [ ]:
import sys
import subprocess

# --- 1. Patch pipelines.py (idempotent) ---
result = subprocess.run(
    ["grep", "-c", "self.device = 'cpu'", "/workspace/Hunyuan3D-2/hy3dgen/texgen/pipelines.py"],
    capture_output=True, text=True
)
cpu_count = int(result.stdout.strip()) if result.stdout.strip() else 0

if cpu_count > 0:
    !sed -i "s/self\.device = 'cpu'/self.device = 'cuda'/g" /workspace/Hunyuan3D-2/hy3dgen/texgen/pipelines.py
    !sed -i 's/self\.device = "cpu"/self.device = "cuda"/g' /workspace/Hunyuan3D-2/hy3dgen/texgen/pipelines.py
    print(f"Patched {cpu_count} occurrences of device='cpu' → 'cuda'")
else:
    print("pipelines.py already patched (no 'cpu' device references found)")

!grep -n "self.device" /workspace/Hunyuan3D-2/hy3dgen/texgen/pipelines.py

# --- 2. Force-reload modules to pick up patch ---
# If Step 5 imported anything from hy3dgen, the texgen submodules are cached
# with the OLD device='cpu' value. We must clear them before re-importing.
mods_to_remove = [key for key in sys.modules if 'hy3dgen.texgen' in key]
for mod in mods_to_remove:
    del sys.modules[mod]
print(f"Cleared {len(mods_to_remove)} cached texgen modules")

# Suppress VAE fallback warnings (weights are .bin, not .safetensors)
import warnings
warnings.filterwarnings("ignore", message=".*no file named diffusion_pytorch_model.safetensors.*")
warnings.filterwarnings("ignore", message=".*Defaulting to unsafe serialization.*")

# --- 3. Import and instantiate ---
from hy3dgen.texgen import Hunyuan3DPaintPipeline

paint_pipeline = Hunyuan3DPaintPipeline.from_pretrained("tencent/Hunyuan3D-2")

# --- 4. Deep device override ---
# Setting paint_pipeline.device = 'cuda' alone is NOT enough.
# Internal sub-components (UNet, VAE, x4 upscaler) each hold their own
# device references. We must walk all nn.Module attributes and move them.
import torch

if hasattr(paint_pipeline, 'device'):
    paint_pipeline.device = 'cuda'

for attr_name in dir(paint_pipeline):
    try:
        attr = getattr(paint_pipeline, attr_name, None)
        if isinstance(attr, torch.nn.Module):
            attr.to('cuda')
            print(f"  Moved {attr_name} to CUDA")
    except Exception:
        pass

if hasattr(paint_pipeline, 'worker'):
    worker = paint_pipeline.worker
    if hasattr(worker, 'device'):
        worker.device = 'cuda'
    for attr_name in dir(worker):
        try:
            attr = getattr(worker, attr_name, None)
            if isinstance(attr, torch.nn.Module):
                attr.to('cuda')
                print(f"  Moved worker.{attr_name} to CUDA")
        except Exception:
            pass

print(f"\nPaint pipeline device: {getattr(paint_pipeline, 'device', 'unknown')}")
if hasattr(paint_pipeline, 'worker'):
    print(f"Worker device: {getattr(paint_pipeline.worker, 'device', 'unknown')}")
print(f"GPU memory before texgen: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")

# --- 5. Run texture generation ---
textured_mesh = paint_pipeline(mesh, image=image)
print("Texture applied successfully.")

## Step 7: Export 3D model

In [ ]:
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
textured_mesh.export(OUTPUT_PATH)
print(f"3D model saved to: {OUTPUT_PATH}")
print(f"File size: {os.path.getsize(OUTPUT_PATH) / 1024 / 1024:.2f} MB")

## Step 8: Visualize

In [ ]:
import numpy as np
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

vertices = np.array(textured_mesh.vertices)
faces = np.array(textured_mesh.faces)

fig = plt.figure(figsize=(10, 10))
ax = fig.add_subplot(111, projection="3d")

max_faces = 5000
if len(faces) > max_faces:
    indices = np.random.choice(len(faces), max_faces, replace=False)
    sampled_faces = faces[indices]
else:
    sampled_faces = faces

poly3d = [[vertices[vert] for vert in face] for face in sampled_faces]
ax.add_collection3d(Poly3DCollection(poly3d, alpha=0.5, edgecolor="k", linewidths=0.1))

scale = vertices.ptp(axis=0).max() / 2
mid = vertices.mean(axis=0)
ax.set_xlim(mid[0] - scale, mid[0] + scale)
ax.set_ylim(mid[1] - scale, mid[1] + scale)
ax.set_zlim(mid[2] - scale, mid[2] + scale)
ax.set_title("Generated 3D Mesh Preview")
plt.tight_layout()
plt.show()

## Step 9: Download model

In [ ]:
from IPython.display import FileLink
FileLink(OUTPUT_PATH)

## Step 10: Free GPU memory

In [ ]:
del shape_pipeline, paint_pipeline
torch.cuda.empty_cache()
print(f"VRAM used after cleanup: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")